In [2]:
import requests
import json
import pandas as pd
from pathlib import Path

# Project paths
BASE_PATH  = Path(r"C:\Data Science\DS Projects\Internet Traffic\Datasets\ITU")
RAW_PATH   = BASE_PATH / "Raw"
CLEAN_PATH = BASE_PATH / "Clean"

RAW_PATH.mkdir(parents=True, exist_ok=True)
CLEAN_PATH.mkdir(parents=True, exist_ok=True)

print("Raw path:  ", RAW_PATH)
print("Clean path:", CLEAN_PATH)

Raw path:   C:\Data Science\DS Projects\Internet Traffic\Datasets\ITU\Raw
Clean path: C:\Data Science\DS Projects\Internet Traffic\Datasets\ITU\Clean


In [3]:
# Fetch internet users data from World Bank API
# Source: ITU via World Bank (indicator IT.NET.USER.ZS)
# WLD = global aggregate, 1990-2023, JSON format

WB_INTERNET_URL = (
    "https://api.worldbank.org/v2/country/WLD/indicator/IT.NET.USER.ZS"
    "?format=json&per_page=100&date=1990:2023"
)

r_internet = requests.get(WB_INTERNET_URL, timeout=10)

print("Status code:", r_internet.status_code)
print("Preview:", r_internet.text[:500])

Status code: 200
Preview: [{"page":1,"pages":1,"per_page":100,"total":34,"sourceid":"2","lastupdated":"2026-04-08"},[{"indicator":{"id":"IT.NET.USER.ZS","value":"Individuals using the Internet (% of population)"},"country":{"id":"1W","value":"World"},"countryiso3code":"WLD","date":"2023","value":69.2,"unit":"","obs_status":"","decimal":0},{"indicator":{"id":"IT.NET.USER.ZS","value":"Individuals using the Internet (% of population)"},"country":{"id":"1W","value":"World"},"countryiso3code":"WLD","date":"2022","value":67,"u


In [4]:
# Save raw JSON response to Raw folder
# Unmodified — exactly as received from the API

raw_internet_path = RAW_PATH / "itu_internet_pct_raw.json"

with open(raw_internet_path, "w") as f:
    json.dump(r_internet.json(), f, indent=2)

print("Saved:", raw_internet_path)

Saved: C:\Data Science\DS Projects\Internet Traffic\Datasets\ITU\Raw\itu_internet_pct_raw.json


In [5]:
# Parse the raw JSON response into a pandas DataFrame
# The API returns data newest-first, so we sort ascending by year

data = r_internet.json()[1]  # [1] skips the metadata, grabs just the records

df = pd.DataFrame([
    {"year": int(rec["date"]), "internet_pct": rec["value"]}
    for rec in data
    if rec["value"] is not None
]).sort_values("year").reset_index(drop=True)

print(f"Rows: {len(df)}")
print(f"Years: {df['year'].min()} – {df['year'].max()}")
df

Rows: 19
Years: 2005 – 2023


,year,internet_pct
0,2005,15.6
1,2006,17.2
2,2007,20.2
3,2008,22.8
4,2009,25.3
5,2010,28.4
6,2011,30.9
7,2012,33.3
8,2013,35.3
9,2014,37.4


In [6]:
# Fetch world population data from World Bank API
# Indicator SP.POP.TOTL = total population, global aggregate (WLD)

WB_POP_URL = (
    "https://api.worldbank.org/v2/country/WLD/indicator/SP.POP.TOTL"
    "?format=json&per_page=100&date=2005:2023"
)

r_population = requests.get(WB_POP_URL, timeout=10)

print("Status code:", r_population.status_code)
print("Preview:", r_population.text[:500])

Status code: 200
Preview: [{"page":1,"pages":1,"per_page":100,"total":19,"sourceid":"2","lastupdated":"2026-04-08"},[{"indicator":{"id":"SP.POP.TOTL","value":"Population, total"},"country":{"id":"1W","value":"World"},"countryiso3code":"WLD","date":"2023","value":8064057930,"unit":"","obs_status":"","decimal":0},{"indicator":{"id":"SP.POP.TOTL","value":"Population, total"},"country":{"id":"1W","value":"World"},"countryiso3code":"WLD","date":"2022","value":7989545217,"unit":"","obs_status":"","decimal":0},{"indicator":{"id


In [7]:
# Save raw population JSON to Raw folder

raw_pop_path = RAW_PATH / "wb_population_raw.json"

with open(raw_pop_path, "w") as f:
    json.dump(r_population.json(), f, indent=2)

print("Saved:", raw_pop_path)

Saved: C:\Data Science\DS Projects\Internet Traffic\Datasets\ITU\Raw\wb_population_raw.json


In [8]:
# Parse population data into a DataFrame
pop_data = r_population.json()[1]

df_pop = pd.DataFrame([
    {"year": int(rec["date"]), "population": rec["value"]}
    for rec in pop_data
    if rec["value"] is not None
]).sort_values("year").reset_index(drop=True)

# Merge on year, then derive absolute user count
df = pd.merge(df, df_pop, on="year")

df["internet_users_millions"] = ((df["internet_pct"] / 100) * df["population"] / 1e6).round(1)

df

,year,internet_pct,population,internet_users_millions
0,2005,15.6,6575841506,1025.8
1,2006,17.2,6659977025,1145.5
2,2007,20.2,6744489399,1362.4
3,2008,22.8,6830513541,1557.4
4,2009,25.3,6916589116,1749.9
5,2010,28.4,7001266876,1988.4
6,2011,30.9,7087120034,2189.9
7,2012,33.3,7176929184,2389.9
8,2013,35.3,7265892332,2564.9
9,2014,37.4,7354068030,2750.4


In [9]:
# Save clean merged DataFrame to Clean folder

clean_path = CLEAN_PATH / "itu_internet_users_2005_2023_clean.csv"

df.to_csv(clean_path, index=False)

print("Saved:", clean_path)

Saved: C:\Data Science\DS Projects\Internet Traffic\Datasets\ITU\Clean\itu_internet_users_2005_2023_clean.csv
